In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

NOTEBOOK 02: DOCUMENT PROCESSING & OCR QUALITY ASSESSMENT
==========================================================

Workshop: Knowledge Management dengan LLM untuk OMBUDSMAN RI

Durasi: 45 menit (10.30-11.15)

oleh: Assoc.Prof.Dr.Eng. Farrikh Alzami, M.Kom


# SECTION 1: SETUP & INSTALASI

In [ ]:
import sys
print("Installing required packages...")

requirements = [
    "sentence-transformers",
    "nltk",
    "python-Levenshtein",
    "langdetect",
    "textstat"
]

for package in requirements:
    print(f"Installing {package}...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\nInstalasi selesai!")


# SECTION 2: IMPORT LIBRARIES

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Text processing
import Levenshtein
from langdetect import detect, LangDetectException
import textstat

# Create output directories
os.makedirs('output_tables', exist_ok=True)
os.makedirs('output_figures', exist_ok=True)

# Visualization settings
plt.style.use('default')
sns.set_palette("husl")

print("Libraries loaded successfully!")


# SECTION 2B: FILE LOADING UTILITIES

Utilities untuk load documents dari file system

In [ ]:
def load_document_from_file(filepath: str) -> Dict:
    """
    Load document dari .txt file
    
    File format:
    TITLE: [title]
    CATEGORY: [category]
    QUALITY: [quality]
    DOC_ID: [doc_id]
    ====...====
    
    [content]
    
    Returns:
        Dict dengan keys: title, category, quality, doc_id, content, filepath
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Parse metadata (first 4 lines)
    metadata = {}
    for i in range(min(4, len(lines))):
        line = lines[i].strip()
        if ':' in line:
            key, value = line.split(':', 1)
            metadata[key.strip().lower()] = value.strip()
    
    # Find separator line (====)
    separator_idx = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('==='):
            separator_idx = i
            break
    
    # Content is everything after separator
    content = ''.join(lines[separator_idx + 2:])  # Skip separator and empty line
    
    return {
        'title': metadata.get('title', 'Unknown'),
        'category': metadata.get('category', 'Unknown'),
        'quality': metadata.get('quality', 'Unknown'),
        'doc_id': metadata.get('doc_id', 'Unknown'),
        'content': content.strip(),
        'filepath': filepath
    }


def load_documents_from_folder(folder_path: str, 
                               category_filter: str = None,
                               quality_filter: str = None) -> List[Dict]:
    """
    Load semua documents dari folder
    
    Args:
        folder_path: Path ke folder containing .txt files
        category_filter: Optional filter by category (e.g., 'Peraturan')
        quality_filter: Optional filter by quality (e.g., 'HIGH')
    
    Returns:
        List of document dicts
    """
    documents = []
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return documents
    
    # Get all .txt files
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    print(f"Found {len(txt_files)} .txt files in {folder_path}")
    
    for filename in sorted(txt_files):
        filepath = os.path.join(folder_path, filename)
        
        try:
            doc = load_document_from_file(filepath)
            
            # Apply filters
            if category_filter and doc['category'] != category_filter:
                continue
            
            if quality_filter and doc['quality'] != quality_filter:
                continue
            
            documents.append(doc)
            
        except Exception as e:
            print(f"Warning: Could not load {filename}: {e}")
    
    return documents

print("File loading utilities defined!")

# SECTION 2C: LOAD SYNTHETIC DOCUMENTS

Load synthetic documents dari folder

KAGGLE USERS:
1. Upload folder 'synthetic_docs_txt' ke Kaggle:
   - Click 'Add Data' > 'Upload'
   - Upload the folder as dataset
   - Note the path (usually /kaggle/input/[dataset-name]/)

2. Update DOCS_FOLDER path below ke actual Kaggle path

LOCAL USERS:
- Update DOCS_FOLDER ke local path dimana synthetic_docs_txt berada

In [ ]:
# IMPORTANT: Update this path sesuai environment
# For Kaggle: /kaggle/input/ombudsman-synthetic-docs/synthetic_docs_txt
# For Local: ./synthetic_docs_txt or absolute path

# Try common paths
possible_paths = [
    '/kaggle/input/ombudsman-synthetic-docs/ombudsman-synthetic-docs/',
]

DOCS_FOLDER = None
for path in possible_paths:
    if os.path.exists(path):
        DOCS_FOLDER = path
        break

if DOCS_FOLDER is None:
    print("ERROR: Could not find synthetic_docs_txt folder!")
    print("Please update DOCS_FOLDER variable dengan correct path.")
    print("\nSearched paths:")
    for p in possible_paths:
        print(f"  - {p}")
else:
    print(f"Documents folder found: {DOCS_FOLDER}")
    
    # List files untuk verification
    files = os.listdir(DOCS_FOLDER)
    print(f"Total files: {len(files)}")
    print("\nSample files:")
    for f in sorted(files)[:5]:
        print(f"  - {f}")

# SECTION 3: DOCUMENT QUALITY METRICS

Quality Assessment Framework
-----------------------------
Kita akan build scoring system untuk evaluate document quality

Metrics:
1. Character Validity (0-1): Berapa % karakter yang valid
2. Word Validity (0-1): Berapa % kata yang ada di kamus
3. Sentence Structure (0-1): Apakah struktur kalimat masuk akal
4. Layout Consistency (0-1): Apakah formatting konsisten
5. Language Confidence (0-1): Confidence language detection


In [ ]:
class DocumentQualityAssessor:
    """
    Assess kualitas dokumen hasil OCR atau extraction
    
    Scoring:
    - 0.9 - 1.0: Excellent (ready untuk RAG)
    - 0.7 - 0.9: Good (minor cleaning needed)
    - 0.5 - 0.7: Fair (significant preprocessing needed)
    - 0.0 - 0.5: Poor (perlu re-OCR atau manual review)
    """
    
    def __init__(self):
        # Indonesian common words untuk validation (simplified)
        # Dalam production, gunakan proper Indonesian dictionary
        self.indonesian_common_words = {
            'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan',
            'adalah', 'ini', 'itu', 'dalam', 'akan', 'telah', 'oleh',
            'sebagai', 'atau', 'tidak', 'dapat', 'ada', 'juga', 'sudah',
            'karena', 'tentang', 'saat', 'antara', 'setiap', 'harus',
            'peraturan', 'undang', 'pasal', 'ayat', 'nomor', 'tahun',
            'republik', 'indonesia', 'presiden', 'menteri', 'pemerintah',
            'pelayanan', 'publik', 'masyarakat', 'pengaduan', 'ombudsman',
            'laporan', 'investigasi', 'rekomendasi', 'prosedur', 'standar'
        }
    
    def assess_character_validity(self, text: str) -> float:
        """
        Hitung persentase karakter yang valid
        Invalid chars: control chars, excessive special symbols
        """
        if not text:
            return 0.0
        
        # Valid characters untuk Indonesian legal documents
        valid_pattern = re.compile(r'[a-zA-Z0-9\s.,;:()\-/\'"àáâãèéêìíîòóôùúû]')
        
        valid_chars = sum(1 for char in text if valid_pattern.match(char))
        total_chars = len(text)
        
        score = valid_chars / total_chars if total_chars > 0 else 0.0
        return round(score, 4)
    
    def assess_word_validity(self, text: str) -> float:
        """
        Hitung persentase kata yang valid
        Valid = ada di common words atau format tertentu (nomor, tanggal, dll)
        """
        words = re.findall(r'\b\w+\b', text.lower())
        
        if not words:
            return 0.0
        
        valid_words = 0
        for word in words:
            # Check against common Indonesian words
            if word in self.indonesian_common_words:
                valid_words += 1
            # Check if it's a number
            elif word.isdigit():
                valid_words += 1
            # Check if it's a reasonable length word (3-15 chars)
            elif 3 <= len(word) <= 15:
                # Check if it has reasonable char distribution
                if self._has_reasonable_char_distribution(word):
                    valid_words += 1
        
        score = valid_words / len(words)
        return round(score, 4)
    
    def _has_reasonable_char_distribution(self, word: str) -> bool:
        """
        Check apakah kata punya distribusi karakter yang masuk akal
        Kata yang buruk: 'aaaaaa', 'xzxzxz', 'qwerty'
        """
        if len(word) < 3:
            return True
        
        # Check for excessive repetition
        char_counts = Counter(word)
        max_count = max(char_counts.values())
        
        # Jika satu karakter muncul > 50% dari panjang kata, suspicious
        if max_count > len(word) * 0.5:
            return False
        
        # Check for vowel presence (Indonesian words usually have vowels)
        vowels = set('aeiouàáâãèéêìíîòóôùúû')
        has_vowel = any(c in vowels for c in word.lower())
        
        return has_vowel
    
    def assess_sentence_structure(self, text: str) -> float:
        """
        Assess struktur kalimat
        Good sentence: proper capitalization, punctuation, length
        """
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if not sentences:
            return 0.0
        
        valid_sentences = 0
        
        for sentence in sentences:
            # Sentence should have reasonable length
            words = sentence.split()
            if 3 <= len(words) <= 50:  # Not too short, not too long
                # Should start with capital letter
                if sentence[0].isupper():
                    # Should have reasonable word distribution
                    avg_word_length = sum(len(w) for w in words) / len(words)
                    if 3 <= avg_word_length <= 12:
                        valid_sentences += 1
        
        score = valid_sentences / len(sentences) if sentences else 0.0
        return round(score, 4)
    
    def assess_layout_consistency(self, text: str) -> float:
        """
        Assess konsistensi layout
        Check: spacing, line breaks, indentation
        """
        lines = text.split('\n')
        
        if len(lines) < 2:
            return 1.0  # Single line, no layout issue
        
        # Check for excessive empty lines
        empty_lines = sum(1 for line in lines if not line.strip())
        empty_ratio = empty_lines / len(lines)
        
        # Check for consistent spacing
        words_per_line = [len(line.split()) for line in lines if line.strip()]
        
        if not words_per_line:
            return 0.0
        
        # Reasonable line length variance
        avg_words = np.mean(words_per_line)
        std_words = np.std(words_per_line)
        
        # Lower variance = more consistent
        consistency_score = 1.0 - min(std_words / (avg_words + 1), 1.0)
        
        # Penalty for too many empty lines
        layout_score = consistency_score * (1.0 - empty_ratio * 0.5)
        
        return round(max(layout_score, 0.0), 4)
    
    def assess_language_confidence(self, text: str) -> float:
        """
        Detect language dan confidence level
        Expected: Indonesian with high confidence
        """
        try:
            # Try to detect language
            detected_lang = detect(text)
            
            # We expect Indonesian (id)
            if detected_lang == 'id':
                # High confidence jika detect sebagai Indonesian
                return 1.0
            else:
                # Lower score if detected as other language
                # (might be mixed or poor quality)
                return 0.6
                
        except LangDetectException:
            # Cannot detect language = very poor quality
            return 0.0
    
    def calculate_overall_score(self, text: str) -> Dict[str, float]:
        """
        Calculate comprehensive quality score
        
        Returns:
            Dict dengan individual scores dan overall score
        """
        scores = {
            'character_validity': self.assess_character_validity(text),
            'word_validity': self.assess_word_validity(text),
            'sentence_structure': self.assess_sentence_structure(text),
            'layout_consistency': self.assess_layout_consistency(text),
            'language_confidence': self.assess_language_confidence(text)
        }
        
        # Weighted average (some metrics more important than others)
        weights = {
            'character_validity': 0.15,
            'word_validity': 0.30,
            'sentence_structure': 0.25,
            'layout_consistency': 0.15,
            'language_confidence': 0.15
        }
        
        overall = sum(scores[k] * weights[k] for k in scores.keys())
        scores['overall_quality'] = round(overall, 4)
        
        return scores

# Initialize assessor
assessor = DocumentQualityAssessor()

print("DocumentQualityAssessor initialized!")
print("\nQuality Score Interpretation:")
print("  0.9 - 1.0: Excellent - Ready untuk RAG")
print("  0.7 - 0.9: Good - Minor cleaning needed")
print("  0.5 - 0.7: Fair - Significant preprocessing needed")
print("  0.0 - 0.5: Poor - Re-OCR atau manual review\n")


# SECTION 4: LOAD DEMO DOCUMENTS (FROM FILES)

Demo: Assess Different Quality Levels
--------------------------------------
Kita akan load 3 Peraturan documents dengan quality berbeda:
1. HIGH QUALITY: Clean digital-born text
2. MEDIUM QUALITY: OCR dengan beberapa error
3. LOW QUALITY: OCR buruk dengan banyak error


In [ ]:
print("\n" + "="*80)
print("LOADING DEMO DOCUMENTS FROM FILES")
print("="*80 + "\n")

# Load 1 Peraturan dari each quality level
demo_docs = []

for quality in ["HIGH", "MEDIUM", "LOW"]:
    docs = load_documents_from_folder(
        DOCS_FOLDER,
        category_filter="Peraturan",
        quality_filter=quality
    )
    
    if docs:
        demo_docs.append(docs[0])  # Take first match
        print(f"Loaded {quality} quality: {docs[0]['doc_id']} - {docs[0]['title'][:50]}...")
    else:
        print(f"Warning: No {quality} quality Peraturan found")

print(f"\nTotal demo documents loaded: {len(demo_docs)}")

# Extract content untuk assessment
if len(demo_docs) >= 3:
    sample_high_quality = demo_docs[0]['content']
    sample_medium_quality = demo_docs[1]['content']
    sample_low_quality = demo_docs[2]['content']
    
    print("\nDocuments ready for quality assessment!")
    print("\nDocument Details:")
    for i, doc in enumerate(demo_docs, 1):
        print(f"\n{i}. {doc['quality']} Quality")
        print(f"   File: {os.path.basename(doc['filepath'])}")
        print(f"   Title: {doc['title'][:60]}...")
        print(f"   Length: {len(doc['content'])} chars, {len(doc['content'].split())} words")
else:
    print("\nERROR: Could not load sufficient demo documents")
    print("Please check that synthetic_docs_txt folder contains Peraturan files")


# SECTION 5: QUALITY ASSESSMENT DEMO

In [ ]:
print("\n" + "="*80)
print("DEMO: QUALITY ASSESSMENT PADA BERBAGAI DOKUMEN")
print("="*80 + "\n")

# Prepare samples dict untuk compatibility
samples = {
    "HIGH QUALITY (Digital-born)": sample_high_quality,
    "MEDIUM QUALITY (Minor OCR errors)": sample_medium_quality,
    "LOW QUALITY (Major OCR errors)": sample_low_quality
}

results = []

for label, text in samples.items():
    print(f"\n{'='*80}")
    print(f"ASSESSING: {label}")
    print(f"{'='*80}\n")
    
    scores = assessor.calculate_overall_score(text)
    
    print("Individual Metrics:")
    print(f"  Character Validity:   {scores['character_validity']:.4f}")
    print(f"  Word Validity:        {scores['word_validity']:.4f}")
    print(f"  Sentence Structure:   {scores['sentence_structure']:.4f}")
    print(f"  Layout Consistency:   {scores['layout_consistency']:.4f}")
    print(f"  Language Confidence:  {scores['language_confidence']:.4f}")
    print(f"\n  OVERALL QUALITY:      {scores['overall_quality']:.4f}")
    
    # Interpretation
    overall = scores['overall_quality']
    if overall >= 0.9:
        interpretation = "EXCELLENT - Ready untuk RAG"
    elif overall >= 0.7:
        interpretation = "GOOD - Minor cleaning needed"
    elif overall >= 0.5:
        interpretation = "FAIR - Significant preprocessing needed"
    else:
        interpretation = "POOR - Re-OCR atau manual review"
    
    print(f"  {interpretation}\n")
    
    # Store for visualization
    results.append({
        'Document': label,
        **scores,
        'Interpretation': interpretation
    })

# SECTION 5: VISUALIZATION - QUALITY COMPARISON

In [ ]:
print("\n" + "="*80)
print("VISUALISASI: QUALITY METRICS COMPARISON")
print("="*80 + "\n")

# Create DataFrame
df_results = pd.DataFrame(results)

# Plot 1: Overall Quality Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot - Overall Quality
ax1 = axes[0]
colors_map = {
    'HIGH QUALITY (Digital-born)': '#2ecc71',
    'MEDIUM QUALITY (Minor OCR errors)': '#f39c12',
    'LOW QUALITY (Major OCR errors)': '#e74c3c'
}
colors = [colors_map[doc] for doc in df_results['Document']]

bars = ax1.barh(df_results['Document'], df_results['overall_quality'], color=colors)
ax1.set_xlabel('Overall Quality Score', fontsize=12, fontweight='bold')
ax1.set_title('Document Quality Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 1.0)
ax1.axvline(x=0.9, color='green', linestyle='--', alpha=0.5, label='Excellent (0.9+)')
ax1.axvline(x=0.7, color='orange', linestyle='--', alpha=0.5, label='Good (0.7+)')
ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Fair (0.5+)')
ax1.legend(loc='lower right')
ax1.grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, df_results['overall_quality'])):
    ax1.text(value + 0.02, bar.get_y() + bar.get_height()/2, 
             f'{value:.3f}', va='center', fontsize=11, fontweight='bold')

# Plot 2: Radar chart - Individual Metrics
ax2 = axes[1]

categories = ['Character\nValidity', 'Word\nValidity', 'Sentence\nStructure', 
              'Layout\nConsistency', 'Language\nConfidence']
num_vars = len(categories)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

# Plot for each document
for idx, row in df_results.iterrows():
    values = [
        row['character_validity'],
        row['word_validity'],
        row['sentence_structure'],
        row['layout_consistency'],
        row['language_confidence']
    ]
    values += values[:1]  # Complete the circle
    
    ax2.plot(angles, values, 'o-', linewidth=2, 
             label=row['Document'], color=colors_map[row['Document']])
    ax2.fill(angles, values, alpha=0.15, color=colors_map[row['Document']])

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories, size=9)
ax2.set_ylim(0, 1)
ax2.set_title('Individual Metrics Breakdown', fontsize=14, fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save figure
fig_path = 'output_figures/quality_metrics_comparison.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {fig_path}")

print("\n KEY INSIGHTS:")
print("-"*80)
print(" HIGH QUALITY documents (digital-born):")
print("  - All metrics > 0.9")
print("  - Ready untuk RAG tanpa preprocessing")
print()
print(" MEDIUM QUALITY documents (minor OCR errors):")
print("  - Word validity turun (typos seperti '0' vs 'O', 'l' vs '1')")
print("  - Layout still reasonable")
print("  - BUTUH: Text correction sebelum masuk RAG")
print()
print(" LOW QUALITY documents (major OCR errors):")
print("  - Multiple metrics < 0.5")
print("  - Layout rusak (excessive spacing)")
print("  - BUTUH: Re-OCR dengan settings lebih baik atau manual review")
print()
print(" ACTIONABLE:")
print("   Sebelum masuk RAG, ALWAYS run quality assessment!")
print("   Documents dengan score < 0.7 harus di-preprocess dulu.\n")

# SECTION 6: TEXT PREPROCESSING PIPELINE

Preprocessing Pipeline untuk Fix Common OCR Errors
---------------------------------------------------

In [ ]:
class TextPreprocessor:
    """
    Clean dan normalize text hasil OCR
    
    Steps:
    1. Fix common OCR errors (0/O, 1/l, etc)
    2. Normalize whitespace
    3. Fix encoding issues
    4. Remove noise (repeated chars, excessive punctuation)
    5. Preserve document structure (sections, numbering)
    """
    
    def __init__(self):
        # Common OCR substitutions untuk Indonesian legal docs
        self.ocr_corrections = {
            # Number confusions
            r'\bl\b': '1',  # standalone l → 1
            r'\bO\b': '0',  # standalone O → 0
            
            # Character confusions
            r'rn': 'm',
            r'vv': 'w',
            r'cl': 'd',
            
            # Indonesian specific
            r'lndonesia': 'Indonesia',
            r'pub1ik': 'publik',
            r'1embaga': 'lembaga',
            r'kewenangan': 'kewenangan',
            r'se1anjutnya': 'selanjutnya',
            r'da1am': 'dalam',
            r'ada1ah': 'adalah',
            r'o1eh': 'oleh',
            r'ke1uhan': 'keluhan',
            r'ma1administrasi': 'maladministrasi',
            r'pe1ayanan': 'pelayanan',
            r'peri1aku': 'perilaku',
            r'me1awan': 'melawan',
            r'me1ampaui': 'melampaui',
            r'1ain': 'lain',
            r'ke1a1aian': 'kelalaian',
            r'kua1itas': 'kualitas',
            r'penye1esaian': 'penyelesaian',
            r'akuntabi1itas': 'akuntabilitas',
            r'Pasa1': 'Pasal',
            r'Da1am': 'Dalam',
            
            # Numbers in formal docs
            r'2O24': '2024',
            r'2O23': '2023',
            r'2O22': '2022',
            r'I5': '15',
            r'l5': '15',
        }
    
    def fix_ocr_errors(self, text: str) -> str:
        """
        Apply OCR error corrections
        """
        corrected = text
        
        for pattern, replacement in self.ocr_corrections.items():
            corrected = re.sub(pattern, replacement, corrected)
        
        return corrected
    
    def normalize_whitespace(self, text: str) -> str:
        """
        Normalize spacing dan line breaks
        - Remove excessive spaces
        - Normalize line breaks
        - Keep paragraph structure
        """
        # Replace multiple spaces with single space
        text = re.sub(r' +', ' ', text)
        
        # Replace excessive line breaks (3+) with double line break
        text = re.sub(r'\n{3,}', '\n\n', text)
        
        # Remove spaces at start/end of lines
        lines = text.split('\n')
        lines = [line.strip() for line in lines]
        text = '\n'.join(lines)
        
        return text
    
    def remove_noise(self, text: str) -> str:
        """
        Remove common noise patterns
        """
        # Remove repeated characters (more than 2 times)
        # But preserve intentional repetitions like "..." or "---"
        text = re.sub(r'([^\.\-\s])\1{2,}', r'\1\1', text)
        
        # Fix excessive punctuation
        text = re.sub(r'\.{4,}', '...', text)
        text = re.sub(r'\-{4,}', '---', text)
        
        return text
    
    def process(self, text: str) -> str:
        """
        Full preprocessing pipeline
        """
        # Step 1: Fix OCR errors
        text = self.fix_ocr_errors(text)
        
        # Step 2: Normalize whitespace
        text = self.normalize_whitespace(text)
        
        # Step 3: Remove noise
        text = self.remove_noise(text)
        
        return text.strip()

# Initialize preprocessor
preprocessor = TextPreprocessor()

print("\n" + "="*80)
print("TEXT PREPROCESSING PIPELINE")
print("="*80 + "\n")

# Demo: Apply preprocessing to MEDIUM quality document
print("BEFORE Preprocessing:")
print("-"*80)
print(sample_medium_quality[:300] + "...\n")

processed_medium = preprocessor.process(sample_medium_quality)

print("\nAFTER Preprocessing:")
print("-"*80)
print(processed_medium[:300] + "...\n")

# Re-assess quality after preprocessing
print("\n" + "="*80)
print("QUALITY IMPROVEMENT ANALYSIS")
print("="*80 + "\n")

before_scores = assessor.calculate_overall_score(sample_medium_quality)
after_scores = assessor.calculate_overall_score(processed_medium)

comparison_df = pd.DataFrame({
    'Metric': ['Character Validity', 'Word Validity', 'Sentence Structure', 
               'Layout Consistency', 'Language Confidence', 'OVERALL QUALITY'],
    'Before': [
        before_scores['character_validity'],
        before_scores['word_validity'],
        before_scores['sentence_structure'],
        before_scores['layout_consistency'],
        before_scores['language_confidence'],
        before_scores['overall_quality']
    ],
    'After': [
        after_scores['character_validity'],
        after_scores['word_validity'],
        after_scores['sentence_structure'],
        after_scores['layout_consistency'],
        after_scores['language_confidence'],
        after_scores['overall_quality']
    ]
})

comparison_df['Improvement'] = comparison_df['After'] - comparison_df['Before']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Before'] * 100).round(2)

print(comparison_df.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Before'], width, label='Before', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['After'], width, label='After', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Metrics', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Quality Improvement After Preprocessing', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Metric'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Save figure
fig_path = 'output_figures/quality_improvement.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {fig_path}")

print("\n KEY INSIGHTS:")
print("-"*80)
print(f" Overall Quality improved: {before_scores['overall_quality']:.4f} → {after_scores['overall_quality']:.4f}")
print(f"  (+{(after_scores['overall_quality'] - before_scores['overall_quality']):.4f} points)")
print()
print(" Biggest improvements:")
print(f"  - Word Validity: {comparison_df.loc[1, 'Improvement %']}% increase")
print(f"  - Character Validity: {comparison_df.loc[0, 'Improvement %']}% increase")
print()
print(" RECOMMENDATION:")
if after_scores['overall_quality'] >= 0.7:
    print("    Dokumen NOW READY untuk masuk RAG pipeline!")
else:
    print("     Perlu additional processing atau re-OCR dengan settings lebih baik\n")

# SECTION 7: HANDS-ON EXERCISE

In [ ]:
print("\n" + "="*80)
print(" HANDS-ON EXERCISE")
print("="*80 + "\n")

print("TASK: Process dokumen LOW QUALITY dan improve kualitasnya\n")
print("Steps:")
print("1. Run quality assessment pada sample_low_quality")
print("2. Apply preprocessing pipeline")
print("3. Re-assess quality")
print("4. Analyze improvement\n")

print(" YOUR TURN - Run code di bawah ini!\n")

# Exercise code (peserta run ini)
exercise_code = """
# Step 1: Assess original quality
print("Step 1: Original Quality Assessment")
print("-"*60)
original_scores = assessor.calculate_overall_score(sample_low_quality)
print(f"Overall Quality: {original_scores['overall_quality']:.4f}")

# Step 2: Apply preprocessing
print("\\nStep 2: Applying Preprocessing...")
print("-"*60)
processed_low = preprocessor.process(sample_low_quality)

# Step 3: Re-assess
print("\\nStep 3: Quality After Preprocessing")
print("-"*60)
processed_scores = assessor.calculate_overall_score(processed_low)
print(f"Overall Quality: {processed_scores['overall_quality']:.4f}")

# Step 4: Compare
print("\\nStep 4: Improvement Analysis")
print("-"*60)
improvement = processed_scores['overall_quality'] - original_scores['overall_quality']
improvement_pct = (improvement / original_scores['overall_quality']) * 100

print(f"Quality Improvement: +{improvement:.4f} ({improvement_pct:.2f}%)")

if processed_scores['overall_quality'] >= 0.7:
    print("\\n SUCCESS: Dokumen now ready untuk RAG!")
elif processed_scores['overall_quality'] >= 0.5:
    print("\\n  PARTIAL: Better, tapi perlu additional steps")
else:
    print("\\n CRITICAL: Perlu re-OCR atau manual review")
"""

# Execute exercise
exec(exercise_code)


# SECTION 8: SUMMARY & NEXT STEPS

In [ ]:
print("\n\n" + "="*80)
print(" SUMMARY - APA YANG SUDAH KITA PELAJARI")
print("="*80 + "\n")

summary_points = [
    " ROOT CAUSE Analysis: Kenapa RAG gagal = OCR quality issue",
    " Quality Assessment Framework: 5 metrics untuk evaluate documents",
    " Diagnostic Tools: Identify problem documents before RAG",
    " Preprocessing Pipeline: Fix common OCR errors automatically",
    " Validation: Measure improvement setelah preprocessing",
    " Decision Framework: When to proceed vs when to re-OCR"
]

for point in summary_points:
    print(point)

print("\n" + "="*80)
print(" ACTIONABLE WORKFLOW UNTUK PRODUCTION")
print("="*80 + "\n")

workflow = """
1. INTAKE: Receive dokumen (scan/PDF)
   ↓
2. ASSESS: Run quality assessment
   ↓
3. DECISION TREE:
   - If score >= 0.9: Proceed to RAG 
   - If 0.7 <= score < 0.9: Apply preprocessing → Re-assess
   - If 0.5 <= score < 0.7: Apply preprocessing + manual review
   - If score < 0.5: Re-OCR atau reject
   ↓
4. PREPROCESS: Clean text, fix errors
   ↓
5. VALIDATE: Re-assess quality
   ↓
6. PROCEED: Masuk ke chunking & embedding (Notebook 03)
"""

print(workflow)

print("\n" + "="*80)
print(" NEXT: NOTEBOOK 03 - CHUNKING STRATEGIES")
print("="*80 + "\n")

print("Next notebook akan cover:")
print("  - Semantic chunking untuk preserve context")
print("  - Legal document specific chunking (Pasal/Ayat aware)")
print("  - Chunk size optimization")
print("  - Evaluation metrics untuk chunking quality\n")

print(" Notebook 02 COMPLETE!")
print("Ready untuk break & hands-on practice!\n")